In [5]:
# Fix for BM25-Validation Query Mismatch
# =====================================

import json
import os
import pandas as pd
import random
from tqdm import tqdm

# Set random seed for reproducibility
random.seed(208973249)

# File paths
qrel_file_path = '/workspace/2404170001/inputs/qrels.dev.tsv'
query_file_path = '/workspace/2404170001/inputs/queries.dev.small.tsv'
collection_file_path = '/workspace/2404170001/inputs/collection.tsv'
output_folder = '/workspace/2404170001/validation_2/pyserini_data_fixed'
os.makedirs(output_folder, exist_ok=True)

# Parameters
number_of_sampled_queries = 3200
max_rank = 100
max_doc_char_length = 100_000

print("=== STEP 1: Load and align query sets ===")

# Load qrels to get query IDs that have relevance judgments
def load_qrels(path):
    qids_to_relevant_passageids = {}
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                doc_id = parts[2]
                relevance = float(parts[3])
                if relevance > 0:
                    if qid not in qids_to_relevant_passageids:
                        qids_to_relevant_passageids[qid] = {}
                    qids_to_relevant_passageids[qid][doc_id] = relevance
    return qids_to_relevant_passageids

# Load all queries
def load_queries(path):
    queries = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                qid, query_text = parts
                queries[qid] = query_text
    return queries

qrels = load_qrels(qrel_file_path)
all_queries = load_queries(query_file_path)

print(f"Total queries in qrels: {len(qrels)}")
print(f"Total queries in query file: {len(all_queries)}")

# Find intersection - only queries that have both text and relevance judgments
valid_query_ids = list(set(qrels.keys()) & set(all_queries.keys()))
print(f"Queries with both text and qrels: {len(valid_query_ids)}")

# Sample queries for validation (same as paper methodology)
if len(valid_query_ids) < number_of_sampled_queries:
    print(f"Warning: Only {len(valid_query_ids)} valid queries, using all")
    sampled_query_ids = set(valid_query_ids)
else:
    sampled_query_ids = set(random.sample(valid_query_ids, number_of_sampled_queries))

print(f"Sampled {len(sampled_query_ids)} queries for validation")


=== STEP 1: Load and align query sets ===
Total queries in qrels: 55578
Total queries in query file: 55578
Queries with both text and qrels: 55578
Sampled 3200 queries for validation


In [6]:

print("\n=== STEP 2: Create query file for BM25 generation ===")

# Create a query file with ONLY the sampled queries
sampled_query_file = os.path.join(output_folder, 'sampled_queries.tsv')
with open(sampled_query_file, 'w', encoding='utf-8') as f:
    for qid in sampled_query_ids:
        f.write(f"{qid}\t{all_queries[qid]}\n")

print(f"Created sampled query file: {sampled_query_file}")

print("\n=== STEP 3: Convert collection to JSONL ===")

collection_jsonl_path = os.path.join(output_folder, 'collection.jsonl')
print(f"Converting collection to {collection_jsonl_path}...")

with open(collection_file_path, 'r', encoding='utf-8') as in_file, \
     open(collection_jsonl_path, 'w', encoding='utf-8') as out_file:
    for line in tqdm(in_file):
        try:
            pid, passage_text = line.strip().split('\t')
            record = {"id": pid, "contents": passage_text}
            out_file.write(json.dumps(record) + '\n')
        except ValueError:
            print(f"Skipping malformed line: {line.strip()}")

print("\n=== STEP 4: Build Lucene index ===")
index_path = os.path.join(output_folder, 'index')

# Note: Run these commands in your environment
print(f"""
Run these commands in your terminal:

# Build index
python -m pyserini.index.lucene \\
  --collection JsonCollection \\
  --input {output_folder} \\
  --index {index_path} \\
  --generator DefaultLuceneDocumentGenerator \\
  --threads 8

# Generate BM25 results with SAMPLED queries only
python -m pyserini.search.lucene \\
  --index {index_path} \\
  --topics {sampled_query_file} \\
  --output {os.path.join(output_folder, 'bm25_run.txt')} \\
  --bm25 \\
  --hits {max_rank} \\
  --output-format trec
""")


=== STEP 2: Create query file for BM25 generation ===
Created sampled query file: /workspace/2404170001/validation_2/pyserini_data_fixed/sampled_queries.tsv

=== STEP 3: Convert collection to JSONL ===
Converting collection to /workspace/2404170001/validation_2/pyserini_data_fixed/collection.jsonl...


8841823it [00:38, 232367.22it/s]


=== STEP 4: Build Lucene index ===

Run these commands in your terminal:

# Build index
python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input /workspace/2404170001/validation_2/pyserini_data_fixed \
  --index /workspace/2404170001/validation_2/pyserini_data_fixed/index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 8

# Generate BM25 results with SAMPLED queries only
python -m pyserini.search.lucene \
  --index /workspace/2404170001/validation_2/pyserini_data_fixed/index \
  --topics /workspace/2404170001/validation_2/pyserini_data_fixed/sampled_queries.tsv \
  --output /workspace/2404170001/validation_2/pyserini_data_fixed/bm25_run.txt \
  --bm25 \
  --hits 100 \
  --output-format trec



In [7]:

print("\n=== STEP 5: Generate validation file (run after BM25) ===")

def generate_validation_file():
    """Generate validation file after BM25 generation is complete"""
    
    bm25_run_file = os.path.join(output_folder, 'bm25_run.txt')
    validation_file = os.path.join(output_folder, 'validation.tsv')
    
    # Load collection
    print("Loading collection...")
    collection = {}
    with open(collection_file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            parts = line.strip().split('\t')
            if len(parts) == 2:
                doc_id, text = parts
                collection[doc_id] = text[:max_doc_char_length]
    
    known_pairs = set()
    
    print("Generating validation file...")
    with open(validation_file, 'w', encoding='utf-8') as out_file:
        
        # Add BM25 candidates
        print("Processing BM25 candidates...")
        if os.path.exists(bm25_run_file):
            with open(bm25_run_file, 'r') as bm25_file:
                for line in tqdm(bm25_file):
                    parts = line.strip().split()
                    if len(parts) >= 4:
                        qid = parts[0]
                        doc_id = parts[2]
                        rank = int(parts[3])
                        
                        if qid in sampled_query_ids and rank <= max_rank:
                            if (qid, doc_id) not in known_pairs:
                                if qid in all_queries and doc_id in collection:
                                    known_pairs.add((qid, doc_id))
                                    row = [qid, doc_id, all_queries[qid], collection[doc_id]]
                                    out_file.write('\t'.join(row) + '\n')
        
        # Add relevant passages from qrels
        print("Adding relevant passages...")
        for qid in tqdm(sampled_query_ids):
            if qid in qrels:
                for doc_id in qrels[qid]:
                    if (qid, doc_id) not in known_pairs:
                        if qid in all_queries and doc_id in collection:
                            known_pairs.add((qid, doc_id))
                            row = [qid, doc_id, all_queries[qid], collection[doc_id]]
                            out_file.write('\t'.join(row) + '\n')
    
    print(f"Validation file created: {validation_file}")
    print(f"Total pairs: {len(known_pairs)}")
    
    return validation_file

# The validation generation function is ready to call after BM25 generation


=== STEP 5: Generate validation file (run after BM25) ===


In [8]:

print("\n=== STEP 6: Validation and verification ===")

def validate_consistency():
    """Validate that BM25 and validation files are consistent"""
    
    bm25_run_file = os.path.join(output_folder, 'bm25_run.txt')
    validation_file = os.path.join(output_folder, 'validation.tsv')
    
    if not os.path.exists(bm25_run_file):
        print("❌ BM25 file not found. Run BM25 generation first.")
        return
    
    if not os.path.exists(validation_file):
        print("❌ Validation file not found. Run validation generation first.")
        return
    
    # Load BM25 results
    bm25_pairs = set()
    with open(bm25_run_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, doc_id = parts[0], parts[2]
                bm25_pairs.add((qid, doc_id))
    
    # Load validation file
    validation_df = pd.read_csv(validation_file, sep='\t', header=None,
                               names=['qid', 'doc_id', 'query', 'doc'])
    validation_pairs = set(zip(validation_df['qid'], validation_df['doc_id']))
    
    # Check consistency
    print(f"BM25 pairs: {len(bm25_pairs)}")
    print(f"Validation pairs: {len(validation_pairs)}")
    print(f"BM25 queries: {len(set(pair[0] for pair in bm25_pairs))}")
    print(f"Validation queries: {len(validation_df['qid'].nunique())}")
    
    # Find mismatches
    bm25_queries = set(pair[0] for pair in bm25_pairs)
    validation_queries = set(validation_df['qid'].unique())
    
    query_mismatch = bm25_queries.symmetric_difference(validation_queries)
    if query_mismatch:
        print(f"❌ Query mismatch found: {len(query_mismatch)} queries differ")
        print(f"Sample mismatched queries: {list(query_mismatch)[:5]}")
    else:
        print("✅ All queries consistent between BM25 and validation")
    
    return len(query_mismatch) == 0

# Store key variables for later use
print(f"\n=== Summary ===")
print(f"Sampled {len(sampled_query_ids)} queries")
print(f"Output folder: {output_folder}")
print(f"Next steps:")
print(f"1. Run the BM25 generation commands above")
print(f"2. Call generate_validation_file() to create validation.tsv")
print(f"3. Call validate_consistency() to verify everything matches")


=== STEP 6: Validation and verification ===

=== Summary ===
Sampled 3200 queries
Output folder: /workspace/2404170001/validation_2/pyserini_data_fixed
Next steps:
1. Run the BM25 generation commands above
2. Call generate_validation_file() to create validation.tsv
3. Call validate_consistency() to verify everything matches


In [11]:
def validate_consistency():
    """Validate that BM25 and validation files are consistent"""
    
    bm25_run_file = os.path.join(output_folder, 'bm25_run.txt')
    validation_file = os.path.join(output_folder, 'validation.tsv')
    
    if not os.path.exists(bm25_run_file):
        print("❌ BM25 file not found. Run BM25 generation first.")
        return
    
    if not os.path.exists(validation_file):
        print("❌ Validation file not found. Run validation generation first.")
        return
    
    # Load BM25 results
    bm25_pairs = set()
    with open(bm25_run_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, doc_id = parts[0], parts[2]
                bm25_pairs.add((qid, doc_id))
    
    # Load validation file
    validation_df = pd.read_csv(validation_file, sep='\t', header=None,
                               names=['qid', 'doc_id', 'query', 'doc'])
    validation_pairs = set(zip(validation_df['qid'], validation_df['doc_id']))
    
    # Check consistency
    print(f"BM25 pairs: {len(bm25_pairs)}")
    print(f"Validation pairs: {len(validation_pairs)}")
    print(f"BM25 queries: {len(set(pair[0] for pair in bm25_pairs))}")
    print(f"Validation queries: {validation_df['qid'].nunique()}")  # Fixed line
    
    # Find mismatches
    bm25_queries = set(pair[0] for pair in bm25_pairs)
    validation_queries = set(validation_df['qid'].unique())
    
    query_mismatch = bm25_queries.symmetric_difference(validation_queries)
    if query_mismatch:
        print(f"❌ Query mismatch found: {len(query_mismatch)} queries differ")
        print(f"Sample mismatched queries: {list(query_mismatch)[:5]}")
    else:
        print("✅ All queries consistent between BM25 and validation")
    
    # Check if all BM25 pairs are in validation
    missing_bm25_pairs = bm25_pairs - validation_pairs
    extra_validation_pairs = validation_pairs - bm25_pairs
    
    print(f"Missing BM25 pairs in validation: {len(missing_bm25_pairs)}")
    print(f"Extra pairs in validation (from qrels): {len(extra_validation_pairs)}")
    
    if len(missing_bm25_pairs) == 0:
        print("✅ All BM25 pairs included in validation")
    else:
        print("❌ Some BM25 pairs missing from validation")
        print(f"Sample missing pairs: {list(missing_bm25_pairs)[:5]}")
    
    return len(query_mismatch) == 0 and len(missing_bm25_pairs) == 0

# Run the corrected validation
validate_consistency()

BM25 pairs: 320000
Validation pairs: 321213
BM25 queries: 3200
Validation queries: 3200
❌ Query mismatch found: 6400 queries differ
Sample mismatched queries: [np.int64(1048578), np.int64(1048582), '10945', '867755', np.int64(98323)]
Missing BM25 pairs in validation: 320000
Extra pairs in validation (from qrels): 321213
❌ Some BM25 pairs missing from validation
Sample missing pairs: [('748997', '8770765'), ('473504', '8840093'), ('136650', '2808343'), ('498149', '5120398'), ('707330', '5501666')]


False

In [10]:
validate_consistency() 

BM25 pairs: 320000
Validation pairs: 321213
BM25 queries: 3200


TypeError: object of type 'int' has no len()